In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

#Считывание датасета и файла
url = "https://raw.githubusercontent.com/AlekPinPin/itmo_python_tasks/refs/heads/main/train.csv"
df_t = pd.read_csv(url)

#Проверка структуры
#df_t.info()

# Задаем random_state для воспроизводимости
RANDOM_STATE = 2026

features = ["Pclass", "Sex", "Age", "SibSp", "Parch"]
X = df_t[features]
y = df_t["Survived"]

#Разбиение датасета на тренировочную/тестовую выборки
X_train, X_test, y_train, y_test = train_test_split (
    X, y,test_size=0.2,random_state=RANDOM_STATE,stratify=y,)

numeric_features = ["Pclass", "Age", "SibSp", "Parch"]
categorical_features = ["Sex"]

#Предобработка данных
preprocessor = ColumnTransformer(
    transformers=[("numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),]),numeric_features,),
        ("categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),]),
            categorical_features,
        ),])


#Метрики бейзлайна
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)

print("Для измерения качества выбрана метрика F1-score. Классификация бинарная - Выжил пассажир или нет.\nВажно учитывать и precision, и recal. Метрика F1 позволяет получить сбалансированный показатель.\n")
y_pred_dummy = dummy_clf.predict(X_test)
print(f"Бейзлайн — accuracy: {accuracy_score(y_test, y_pred_dummy):.3f}, "
    f"F1: {f1_score(y_test, y_pred_dummy):.3f}")

# Pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
])

#Обучение модели
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Оценка качества модели
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Модель — accuracy: {accuracy:.3f}")
print(f"Модель — precision: {precision:.3f}")
print(f"Модель — recall: {recall:.3f}")
print(f"Модель — F1: {f1:.3f}")
print(f"Модель — ROC-AUC: {roc_auc:.3f}")



Для измерения качества выбрана метрика F1-score. Классификация бинарная - Выжил пассажир или нет.
Важно учитывать и precision, и recal. Метрика F1 позволяет получить сбалансированный показатель.

Бейзлайн — accuracy: 0.615, F1: 0.000
Модель — accuracy: 0.788
Модель — precision: 0.718
Модель — recall: 0.739
Модель — F1: 0.729
Модель — ROC-AUC: 0.849
